# Project 3. Part 1 - CLD Aggregation Tool

Last update: 06/19/2024

Title: Streamlining the Aggregation of Causal Loop Diagrams: A Thematic Labeling and Adjacency Matrix Approach

Overview: 
Spaghetti Causal Loop Diagrams (CLDs) present a significant challenge for modelers in effectively communicating complex system dynamics to stakeholders. Traditionally, modelers manually synthesize multiple CLDs to create an aggregated diagram that provides a higher-level understanding of the system. This paper introduces a streamlined procedure for aggregating CLDs using general categorical labels. 

The methodology involves translating qualitative mental models into adjacency matrices that capture both links and polarity signs, manually coding and substituting variable names with thematic labels, aggregating these matrices to identify significant feedback loops, and validating the approach by comparing it with the manual synthesis performed by an expert system dynamicist. 

Our results demonstrate that this tool is highly useful for group model-building experts, particularly when dealing with a large number of mental maps. The proposed method enhances the clarity and efficiency of synthesizing complex CLDs, thereby improving the overall communication of system dynamics.







In [34]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# READ I/O

In [35]:
from spellchecker import SpellChecker
from pathlib import Path

import numpy as np
import os 
import pickle
import pandas as pd
import sys
import networkx as nx
import matplotlib.pyplot as plt
import gravis as gv



In [36]:

# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

# Now you can import the data_io module from src
import src.data_io as data_io
import src.cld_nx as cld_nx
import src.parse_cld as parse_cld

from src.vensim_parser import CLDParser

# Read Data

In [37]:
print(os.getcwd())


/Users/gl665/PycharmProjects/Phrame_MentalMaps/notebook


## read raw data

In [38]:
# SYNCH 
dataset_folder_path = 'results/SYNCH/JSON_file/RC/archive'

# PHRAME
# dataset_folder_path = 'results/PHRAME/JSON_file/RC'

In [39]:
# Go one level up from the current directory
cwd = os.getcwd()

parent_dir = Path(cwd).parent
print(parent_dir)

my_data= {}
folder_path = str(parent_dir / dataset_folder_path)


cld_data = data_io.read_json(folder_path, my_data)


/Users/gl665/PycharmProjects/Phrame_MentalMaps
load: FM 51220Clean.json
load: Passages Clean.json
load: full circle GBM 5 24 21clean.json
load: C 7 6 20 clean model.json
load: Wavi combined clean.json
load: drug justice model clean.json
load: DSS23.json
load: E clean model 572020.json
load: nat model clean2.json
load: FC Prov Clean.json
load: full circle clean.json
load: reentry clean model.json
load: DV justice model clean.json


## Read Preprocessed data

In [22]:
# # read RC data after cleaning and merging the semantic similar variable names 
# my_data_RC = {}
# file_path = "results/SYNCH/JSON_file/RC_cld_data_cleaned_preprocessed.json"

# merged_file_path = str(parent_dir / file_path)

# # # Read the JSON file
# with open(merged_file_path, 'rb') as file:
#     my_data_RC = pickle.load(file)

# # Print the data
# for key in my_data_RC.keys():
#     print(key)



## Read file_rename excel sheet

In [25]:
# file_path = str(parent_dir / 'results/SYNCH/file_rename/RC_file_rename.xlsx')


# file_rename = pd.read_excel(file_path, index_col=0)
# file_rename['file_no.'] = [i+1 for i in range(len(file_rename.index))]

# dict_file_rename = dict(zip(file_rename['Filename'], file_rename['Model Name']))
# dict_file_rename

{'FM 51220Clean.json': 'FM',
 'Passages Clean.json': 'PA',
 'full circle GBM 5 24 21clean.json': 'FullGMB',
 'C 7 6 20 clean model.json': 'C76',
 'tracy clean model.json': 'Tracy',
 'Wavi combined clean.json': 'Wavi',
 'drug justice model clean.json': 'JUSTICE',
 '7 13 20 clean model.json': 'C713',
 'DSS23.json': 'DSS',
 'E clean model 572020.json': 'E',
 'nat model clean2.json': 'NAT',
 'j model 7 13 20 clean model3.json': 'J713',
 'FC Prov Clean.json': 'FC',
 'full circle clean.json': 'Full',
 'reentry clean model.json': 'Reentry',
 '1 6 21 Legal clean.json': 'Legal',
 'DV justice model clean.json': 'DV',
 'SOC clean.json': 'SOC',
 'RC 4 28 2021 Clean.json': 'RC'}

## Load all the graphs in a dictionary, {cld_name: networkX graph}

In [26]:
dict_cld = {}

for k, v in my_data.items():
    cld_list, G_cld = cld_nx.get_cld_graph(my_data, k)
    # cld_name = dict_file_rename.get(k, "")
    dict_cld[k] = G_cld

list_of_G = list(dict_cld.values())

for k, v in dict_cld.items(): 
    print(k, ":", v)


RC_5_20_2024.json : DiGraph with 95 nodes and 123 edges
RC_4_16_2024.json : DiGraph with 95 nodes and 131 edges
RC_1_20_2024.json : DiGraph with 71 nodes and 84 edges
RC_5_29_2024.json : DiGraph with 114 nodes and 137 edges
RC_4_17_2024.json : DiGraph with 90 nodes and 114 edges
RC_1 3_1_2024.json : DiGraph with 90 nodes and 110 edges


In [23]:
for k, v in dict_cld.items():
    cld_nx.cld_nx_properties(v)

Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 95
Number of edges: 123
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 95
Number of edges: 131
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 71
Number of edges: 84
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 114
Number of edges: 137
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 90
Number of edges: 114
Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 90
Number of edges: 110


In [ ]:
# Assuming you have a list of graphs called list_of_G

# Initialize lists to store the graph properties
num_nodes = []
num_edges = []
densities = []
avg_clustering_coeffs = []
transitivities = []

for graph in list_of_G:
    print(f"Summary statistics for graph: {graph.name if hasattr(graph, 'name') else 'Unnamed graph'}")
    print("Number of nodes:", graph.number_of_nodes())
    print("Number of edges:", graph.number_of_edges())
    # print("Density:", nx.density(graph))
    # print("Average clustering coefficient:", nx.average_clustering(graph))
    # print("Transitivity:", nx.transitivity(graph))
    # print()

    # Append the graph properties to the respective lists
    num_nodes.append(graph.number_of_nodes())
    num_edges.append(graph.number_of_edges())
    densities.append(nx.density(graph))
    avg_clustering_coeffs.append(nx.average_clustering(graph))
    transitivities.append(nx.transitivity(graph))

# Calculate the summary statistics for each graph property
print("Summary statistics across all graphs:")
print("Number of nodes: Mean = {:.3f}".format(statistics.mean(num_nodes)), "Max = {:.3f}".format(max(num_nodes)), "Min = {:.3f}".format(min(num_nodes)), "Median = {:.3f}".format(statistics.median(num_nodes)))
print("Number of edges: Mean = {:.3f}".format(statistics.mean(num_edges)), "Max = {:.3f}".format(max(num_edges)), "Min = {:.3f}".format(min(num_edges)), "Median = {:.3f}".format(statistics.median(num_edges)))
# print("Density: Mean = {:.3f}".format(statistics.mean(densities)), "Max = {:.3f}".format(max(densities)), "Min = {:.3f}".format(min(densities)), "Median = {:.3f}".format(statistics.median(densities)))
# print("Average clustering coefficient: Mean = {:.3f}".format(statistics.mean(avg_clustering_coeffs)), "Max = {:.3f}".format(max(avg_clustering_coeffs)), "Min = {:.3f}".format(min(avg_clustering_coeffs)), "Median = {:.3f}".format(statistics.median(avg_clustering_coeffs)))
# print("Transitivity: Mean = {:.3f}".format(statistics.mean(transitivities)), "Max = {:.3f}".format(max(transitivities)), "Min = {:.3f}".format(min(transitivities)), "Median = {:.3f}".format(statistics.median(transitivities)))
# TODO: use our paper measures? 


## Read handcraft CLD data

In [ ]:
# # read handcraft cld data 
# input_file = 'data/SYNCH/handcraft_aggergate_model/AggregateModel_SubsystemColorsfullmodelcolors.mdl'

# file_path = str(parent_dir / input_file)

# df =  data_io.read_cld_file(file_path)
# cld = CLDParser(df)
# cld_hand_craft = cld.get_cld_dict()

# # get networkX form of hand_craft_cld
# G_cld_hand_craft = nx.from_dict_of_dicts(cld_hand_craft[1])
# G_cld_hand_craft = nx.relabel_nodes(G_cld_hand_craft, cld_hand_craft[0])

# # get the variable names used in cld_hand_craft
# list_CLD_hand_craft_var_names = np.sort(list(G_cld_hand_craft.nodes()))


# print("number of cld hand craft edges: ", G_cld_hand_craft.number_of_edges())
# print("number of cld hand craft nodes: ", G_cld_hand_craft.number_of_nodes())
# print(list_CLD_hand_craft_var_names)

## Read Themantic Data excel sheet

In [ ]:
# preprocess the RC_Updated_Themes.xslx and match it back with the RC files. 
input_file = 'data/SYNCH/aggregate_theme/Rapid_City_Updated_Themes_w_levels.xlsx'
file_path = str(parent_dir / input_file)
df_rc_themes = pd.read_excel(file_path, sheet_name= "clean")

# get column 1 and 4
df_theme_names = df_rc_themes.iloc[:, [1, 3, 4, 5 ,6]]
df_theme_names.columns = ['theme', 'sub_theme', 'variable', 'level', 'level_qualifier'] # TODO: use the subthem label when complete 


#  delete rows if all columns are nan 
df_theme_names = df_theme_names.dropna(subset=['theme', 'variable'], how='all')


df_theme_names['theme'] = df_theme_names['theme'].ffill()
df_theme_names['theme']= df_theme_names['theme'].str.lower()
df_theme_names['variable']= df_theme_names['variable'].str.lower()

df_theme_names.head(3)

# df_aggregate_names.to_csv('Dataset/SYNCH/aggregate_theme/RC_theme_var_matching.csv', index=False)

In [ ]:
# create columns for different levels 
df_theme_names['personal'] = df_theme_names['level'].apply(lambda x: 'personal' in str(x).lower())
df_theme_names['institutional'] = df_theme_names['level'].apply(lambda x: 'institutional' in str(x).lower())
df_theme_names['relational'] = df_theme_names['level'].apply(lambda x: 'relational' in str(x).lower())
df_theme_names['cultural'] = df_theme_names['level'].apply(lambda x: 'cultural' in str(x).lower())
df_theme_names['community'] = df_theme_names['level'].apply(lambda x: 'community' in str(x).lower())
df_theme_names['societal'] = df_theme_names['level'].apply(lambda x: 'societal' in str(x).lower())

df_theme_names.head(5)


In [ ]:
#  TODO: Q: what does these levels mean? how should I categorize them? 
df_theme_names['level'].unique()

In [ ]:
# df_aggregate_names = pd.read_csv('data/SYNCH/aggregate_theme/RC_theme_var_matching.csv')
unique_theme_names = set(df_theme_names['theme'])
# df_aggregate_names.head()


In [ ]:
# df_aggregate_names = pd.read_csv('data/SYNCH/aggregate_theme/RC_theme_var_matching.csv')
# unique_theme_names = set(df_aggregate_names['theme'])
# df_aggregate_names.head()


# Read Data Definition Dictionary

In [40]:
input_file = 'data/SYNCH/RC/excel_sheets/Rapid City SYNCH Models data dictionary0619.xlsx'

# TODO: check only Passage and Wavi First 
# Data Dictionary 
# Go one level up from the current directory
cwd = os.getcwd()


file_path = str(parent_dir / input_file)

# Read all sheets into a dictionary of DataFrames
excel_data = pd.read_excel(file_path, sheet_name=None)

# Print the keys (sheet names) of the dictionary
print("Sheet names:", list(excel_data.keys()))

# Access a specific sheet
sheet_name = "WAVI (combined)"
df = excel_data[sheet_name]

print(f"\nData from '{sheet_name}':")
df.head()
# print(f"\nData from '{sheet_name}' sheet:\n{df.head()}")

Sheet names: ['full', 'Passages', 'WAVI (combined)', 'Full Circle (patient combined)', 'DSS (combined)', 'justice- drugs', 'Justice - IPV', 'full circle (provider)', 'E model 5 7 2020', 'FM 5 122020', 'nat model', 'j model 7 13 20', 'c model 7 6 20', 'reentry', 'RC Diversion', 'RC Clean', 'RC Legal', 'RC Full Circle', 'description of sessions', 'Sheet1']

Data from 'WAVI (combined)':


,variable,changed variable name,Definition,notes
0,ACES,adverse childhood experiences,Adverse childhood experiences - can include ch...,NaN
1,Alc M,Alcohol Misuse,using alcohol - in reference to misuse,NaN
2,Approval of therapy,NaN,Percieved acceptability of therapy as a soluti...,NaN
3,Child being abused in home,NaN,From the perspective of the parent - observing...,while ACES refers to one's own experiences as ...
4,Community Alc M Rate,Community Alcohol Misuse Rate,The rate of alcohol misuse within the community,NaN


# Export all variable names of CLDs 


In [41]:
df = pd.DataFrame()
for k, v in dict_cld.items():
    for node in v.nodes:
        model_name = k.split('.')[0]
        temp_df = pd.DataFrame({'variable': list(v.nodes), 'model': [model_name] * len(v.nodes)})
    df = pd.concat([df, temp_df], ignore_index=True)


In [42]:
df.head()

,variable,model
0,people who are pregnant,RC_5_20_2024
1,people's health during pregnancy,RC_5_20_2024
2,people getting care postpartum,RC_5_20_2024
3,people getting care for pregnancy,RC_5_20_2024
4,people giving birth,RC_5_20_2024


In [33]:
# export excel sheet
# file_path = str(parent_dir / 'results/PHRAME/JSON_file/RC/result_sheets/RC_variable_list.xlsx')
# df.to_excel(file_path, index=False)


# Compare List of Graphs variable names and Handcraft variable names 

Check if the two have a overlap. 

Result: there's only 9 variables overlapp between the handcraft CLD v.s. the list of individual graphs. that means the handcraft CLD is a merge of the individual CLDs however, the variable names does not match up exactly. 

Hence, there is no baseline, to compare the handcraft cld with the list of individual clds. 



In [ ]:
# check how much variables overlapped between the two list of all variable names 
overlap = [var for var in all_var_names if var in list_CLD_hand_craft_var_names]
print("Overlapping variables between all the variables names used in the RC models and the handcraft model: ", len(overlap))


for i, name in enumerate(overlap, start=1):
    print(f"{i}. {name}")



# Add attributes of the levels to the nodes based on the themantic label

In [ ]:
# set node and cld attributes 
for k , v in dict_cld.items():
    nx.set_node_attributes(v, name = 'cld', values = k)
    nx.set_edge_attributes(v, name = 'cld', values = k)
    cld_nx.set_node_theme_attributes(v, unique_theme_names)


    

In [ ]:
def set_level_node_attribute(G, df_aggregate_names, level: str):
    mydict = {row['variable']: row[level] for _, row in df_aggregate_names.iterrows()}

    for node in G.nodes():
        if node in mydict:
            G.nodes[node][level] = 1 if mydict[node] else 0

    return G

for k, v in dict_cld.items():
    G1 = v

    G1 = set_level_node_attribute(G1, df_theme_names,  'personal')

    G1 = set_level_node_attribute(G1, df_theme_names,  'institutional')

    G1 = set_level_node_attribute(G1, df_theme_names,  'relational')

    G1 = set_level_node_attribute(G1, df_theme_names,  'cultural')

    G1 = set_level_node_attribute(G1, df_theme_names,  'community')




In [ ]:
# merge the node name based on the theme dict 

# for k , v in dict_cld.items():
#     v = nx.relabel_nodes(v, theme_dict)

In [ ]:
dict_cld

In [ ]:
cld_nx.cld_nx_properties(dict_cld['FM 51220Clean.json'])
cld_nx.cld_nx_node_properties(dict_cld['FM 51220Clean.json'])
cld_nx.cld_nx_edge_properties(dict_cld['FM 51220Clean.json'])

In [ ]:
dict_cld['FM 51220Clean.json'] = cld_nx.clean_graph_for_gravis(dict_cld['FM 51220Clean.json'])
cld_nx.gravis_settings(dict_cld['FM 51220Clean.json'])


gv.d3(
    dict_cld['FM 51220Clean.json'], 
    graph_height=500)
#     edge_curvature=-0.5,
    
#     node_size_data_source="degree",
#     use_node_size_normalization=True,
#     node_size_normalization_min=15,
#     node_size_normalization_max=35,
#     show_node_label=True,
#     many_body_force_strength=-500
#  ) 



# Swap the node names to the theme names based on the themantic label 

# Merging by adding up adjency matrix (Self define version)

In [ ]:
# get combined graph                          
combined_graph = cld_nx.combine_graphs_with_node_attributes(list_of_G)

In [ ]:
cld_nx.cld_nx_edge_properties(combined_graph)

In [ ]:
clean_combined_graph = cld_nx.clean_combined_graph(combined_graph, 0, 0 )

cld_nx.cld_nx_properties(clean_combined_graph)
cld_nx.cld_nx_node_properties(clean_combined_graph)
cld_nx.cld_nx_edge_properties(clean_combined_graph)


In [ ]:
# pickle.dump(combined_graph, open('Graph_union.pickle', 'wb'))

cld_nx.gravis_settings(clean_combined_graph)


gv.d3(
    clean_combined_graph, 
    graph_height=600,
    # edge_curvature=-0.5,
    
    # node_size_data_source="degree",
    # use_node_size_normalization=True,
    # node_size_normalization_min=15,
    # node_size_normalization_max=35,
    # show_node_label=True,
    # many_body_force_strength=-500
 ) 

# fig.export_html('Graph_union.html')


In [ ]:
# get trimmed graph 2 overlap nodes 
Graph_1_0 = cld_nx.clean_combined_graph(combined_graph, num_aggregate_nodes = 0, num_aggregate_edges= 0)
cld_nx.cld_nx_properties(Graph_1_0)


In [ ]:
# cld_nx.gravis_settings(Graph_1_0)

gv.d3(
    Graph_1_0, 
    graph_height=500,
    edge_curvature=-0.5,
    
    node_size_data_source="degree",
    use_node_size_normalization=True,
    node_size_normalization_min=15,
    node_size_normalization_max=35,
    show_node_label=True,
    many_body_force_strength=-500
 ) 


In [ ]:
fig = gv.three(Graph_1_0)
fig.export_html('Graph_1_0.html')

In [ ]:
# save graph object to file
# pickle.dump(Graph_1_0, open('Graph_1_0.pickle', 'wb'))

# load graph object from file
# G = pickle.load(open('filename.pickle', 'rb'))

In [ ]:
# get trimmed graph 2 overlap nodes 
Graph_0_1 = cld_nx.clean_combined_graph(combined_graph, num_aggregate_nodes = 0, num_aggregate_edges= 1)
cld_nx.cld_nx_properties(Graph_0_1)

In [ ]:
cld_nx.cld_nx_node_properties(Graph_0_1)

In [ ]:
cld_nx.gravis_settings(Graph_0_1)

fit = gv.d3(
    Graph_0_1, 
    graph_height=500,
    edge_curvature=-0.5,
    
    node_size_data_source="degree",
    use_node_size_normalization=True,
    node_size_normalization_min=15,
    node_size_normalization_max=35,
    show_node_label=True,
    many_body_force_strength=-500
 ) 

fig.export_html('Graph_0_1.html')

# CLD analysis 

Conduct analysis on the merged CLD. 

TODO: 
1. Find loops 
2. Find difference between 
2. Compare the merged CLD with the handcraft CLD
    - Read Arielle's handcraft CLD 
    - compare the shape between the two 


In [ ]:
graph_cycle  = sorted(nx.simple_cycles(Graph_1_0))


In [ ]:
subgraph = Graph_1_0.subgraph(graph_cycle[2]).copy()

edge_weight = cld_nx.mulitiple_weights(subgraph)
for nodes, attrs in subgraph.nodes(data=True):
    print(nodes, attrs)

for u, v, attrs in subgraph.edges(data=True):
    print(u, v, attrs)


In [ ]:
cld_nx.gravis_settings(subgraph)

fig = gv.d3(
    subgraph, 
    graph_height=500,
    edge_curvature=-0.5,
    
    node_size_data_source="degree",
    use_node_size_normalization=True,
    node_size_normalization_min=15,
    node_size_normalization_max=35,
    show_node_label=True,
    many_body_force_strength=-500
 ) 

# fig.export_html('graph_cycle[2].html')

In [ ]:
# subgraph = Graph_1_0.subgraph(graph_cycle[9]).copy()

for i in range(len(graph_cycle)):
    subgraph = Graph_1_0.subgraph(graph_cycle[i]).copy()

    print(subgraph)

    edge_weight = cld_nx.mulitiple_weights(subgraph)
    print(edge_weight)


In [ ]:
graph_cycle[0]

# grab the weights of each cycle
# for u, v in graph_cycle[0]:
#     print(u, v, Graph_1_0[u][v]['weight'])


for u, v , attrs in Graph_1_0.edges(data = True):
    print(u, v, attrs['weight'])
    
    
        
    # print(u, v, Graph_1_0[u][v]['weight'])

# weights = [Graph_1_0[u][v]['weight'] for u, v in graph_cycle[0]]
# weights

In [ ]:
# find cycle 
cycles = nx.find_cycle(Graph_0_1)
cycles

sorted(nx.simple_cycles(Graph_0_1))


In [ ]:
cld_nx.cld_nx_properties(Graph_1_0)
cld_nx.cld_nx_node_properties(Graph_1_0)
cld_nx.cld_nx_edge_properties(Graph_1_0)

In [ ]:
cld_nx.cld_nx_edge_properties(Graph_1_0)

In [ ]:
for name, nx_graph in dict_cld.items(): 
    print(name)
    cycles = nx.find_cycle(nx_graph)
    print(cycles)